# Micheal: trial-level TW × time-cross decoding

首选两层方案：每个 session 内按原始 trial ID 对齐，在 trials 间计算 `TW time × train time × test time` Pearson 相关；session 相关先 Fisher-z 再平均；群体水平对每位被试整张三维图统一 sign-flip，并进行六邻域三维 cluster-mass 校正。

In [1]:
from pathlib import Path
import importlib, sys
import numpy as np
from IPython.display import Image, display

PROJECT_DIR = Path('/home/dilay/project2/tw')
MODULE_DIR = PROJECT_DIR / 'travelling_waves/tw/fft/micheal/corr/cross_corr'
sys.path.insert(0, str(MODULE_DIR))
import michael_tw_cross_correlation as cross_corr
cross_corr = importlib.reload(cross_corr)

TW_DIR = PROJECT_DIR / 'results/micheal_fft'
DECODING_DIR = PROJECT_DIR / 'results/michael_alpha_time_cross_basis_kfold_fir_timepoint_separate_sessions'
OUTPUT_DIR = DECODING_DIR / 'tw_cross_correlation'

MEASURES = ('fw', 'bw')
COMPONENTS = ('difference', 'contra', 'ipsi', 'midline', 'all_lines')
CONDITIONS = ('early', 'late', 'early_minus_late')
FMIN, FMAX = 8, 12
TW_LIMITS = (0.0, 5.7)
TRAIN_LIMITS = (0.0, 6.0)
TEST_LIMITS = (0.0, 6.0)
TW_STRIDE = 1                  # 1 keeps every 50-ms TW sample
DECODING_STRIDE = 1            # 1 keeps every 50-ms decoding sample
N_PERMUTATIONS = 1000
CLUSTER_ALPHA = 0.05
CLUSTER_P = 0.05
SEED = 42

ANALYSES = [
    (measure, component, condition)
    for measure in MEASURES
    for component in COMPONENTS
    for condition in CONDITIONS
]
print('analyses:', ANALYSES)

analyses: [('fw', 'difference', 'early'), ('fw', 'difference', 'late'), ('fw', 'difference', 'early_minus_late'), ('fw', 'contra', 'early'), ('fw', 'contra', 'late'), ('fw', 'contra', 'early_minus_late'), ('fw', 'ipsi', 'early'), ('fw', 'ipsi', 'late'), ('fw', 'ipsi', 'early_minus_late'), ('fw', 'midline', 'early'), ('fw', 'midline', 'late'), ('fw', 'midline', 'early_minus_late'), ('fw', 'all_lines', 'early'), ('fw', 'all_lines', 'late'), ('fw', 'all_lines', 'early_minus_late'), ('bw', 'difference', 'early'), ('bw', 'difference', 'late'), ('bw', 'difference', 'early_minus_late'), ('bw', 'contra', 'early'), ('bw', 'contra', 'late'), ('bw', 'contra', 'early_minus_late'), ('bw', 'ipsi', 'early'), ('bw', 'ipsi', 'late'), ('bw', 'ipsi', 'early_minus_late'), ('bw', 'midline', 'early'), ('bw', 'midline', 'late'), ('bw', 'midline', 'early_minus_late'), ('bw', 'all_lines', 'early'), ('bw', 'all_lines', 'late'), ('bw', 'all_lines', 'early_minus_late')]


## 1. 全时段批量三维相关

对全部 measure × component × condition 组合运行 trial-level 三维相关。每个 trial 的 decoding evidence 是重复 K-fold 后的 held-out trial 平均值；重复次数不会被当作额外 trial，Session 1/2 独立计算相关。

In [2]:
# Run one combination at a time so six large subject-level 3-D arrays are
# never retained in memory simultaneously.
analysis_results = {}
for measure, component, condition in ANALYSES:
    print(f'\n=== {measure.upper()} / {component} / {condition} ===', flush=True)
    subjects, tw_time, train_time, test_time, subject_z, matched_counts = (
        cross_corr.build_subject_cross_maps(
            TW_DIR, DECODING_DIR,
            measure=measure, component=component, condition=condition,
            fmin=FMIN, fmax=FMAX,
            tw_limits=TW_LIMITS, train_limits=TRAIN_LIMITS, test_limits=TEST_LIMITS,
            tw_stride=TW_STRIDE, decoding_stride=DECODING_STRIDE,
        )
    )
    print('subjects:', subjects)
    print('subject Fisher-z maps:', subject_z.shape)

    mean_z, observed_t, clusters, null_max = cross_corr.cluster_signflip_3d(
        subject_z, permutations=N_PERMUTATIONS,
        cluster_alpha=CLUSTER_ALPHA, cluster_p=CLUSTER_P,
        seed=SEED, return_all=True,
    )
    significant = [cluster for cluster in clusters if cluster['significant']]
    print(f'{len(significant)} significant 3-D cluster(s)')

    tag = f'{measure}_{component}_{condition}'
    output_prefix = OUTPUT_DIR / f'tw-cross-3d_{tag}'
    figure_path = output_prefix.with_name(
        output_prefix.name + '_significant-projections.png'
    )
    report = cross_corr.save_analysis(
        output_prefix, subjects, tw_time, train_time, test_time,
        subject_z, mean_z, observed_t, clusters, null_max, matched_counts,
    )
    cross_corr.plot_significant_3d_projections(
        observed_t, clusters, tw_time, train_time, test_time, figure_path,
        title=f'{measure.upper()} TW ({component}) × {condition} time-cross decoding',
    )
    analysis_results[(measure, component, condition)] = {
        'output_prefix': output_prefix,
        'figure_path': figure_path,
        'report': report,
        'n_subjects': len(subjects),
        'n_significant': len(significant),
    }
    display(Image(filename=str(figure_path)))
    del subject_z, mean_z, observed_t, clusters, null_max

print('\nCompleted all analyses.')


=== FW / difference / early ===
subject 01: matched s1=769, s2=770; map=(115, 121, 121)
subject 02: matched s1=826, s2=823; map=(115, 121, 121)
subject 03: matched s1=657, s2=697; map=(115, 121, 121)
subject 04: matched s1=800, s2=789; map=(115, 121, 121)
subject 05: matched s1=787, s2=790; map=(115, 121, 121)
subject 06: matched s1=771, s2=757; map=(115, 121, 121)
subject 07: matched s1=769, s2=771; map=(115, 121, 121)
subject 08: matched s1=775, s2=733; map=(115, 121, 121)
subject 09: matched s1=749, s2=730; map=(115, 121, 121)
subject 10: matched s1=829, s2=823; map=(115, 121, 121)
subject 11: matched s1=772, s2=768; map=(115, 121, 121)
subject 12: matched s1=788, s2=805; map=(115, 121, 121)
subject 13: matched s1=826, s2=808; map=(115, 121, 121)
subject 14: matched s1=674, s2=668; map=(115, 121, 121)
subject 15: matched s1=774, s2=778; map=(115, 121, 121)
subject 16: matched s1=822, s2=817; map=(115, 121, 121)
subject 17: matched s1=820, s2=808; map=(115, 121, 121)
subject 18: mat

KeyboardInterrupt: 

## 2. 探索性同窗 3D correlation（全部 TW components，可单独运行）

在 1.8–3.8 s 的同窗空间内，探索性检验 FW/BW × `difference`、`contra`、`ipsi`、`midline`、`all_lines` × early、late、early−late 的全部组合。每个组合完成完整三维 cluster sign-flip 后立即绘制并保存 3D statistic map 与显著 3D cluster 双面板图。本分析保留 `Training time = Test time` 主对角线；由于同时考察多个方向、component 和 decoding condition，应明确作为探索性分析报告。

In [ ]:
# Standalone exploratory same-window 3-D correlation cell.
from pathlib import Path
import importlib
import sys
from scipy.stats import t as student_t

PROJECT_DIR = Path('/home/dilay/project2/tw')
MODULE_DIR = PROJECT_DIR / 'travelling_waves/tw/fft/micheal/corr/cross_corr'
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import michael_tw_cross_correlation as cross_corr
cross_corr = importlib.reload(cross_corr)

TW_DIR = PROJECT_DIR / 'results/micheal_fft'
DECODING_DIR = PROJECT_DIR / 'results/michael_alpha_time_cross_basis_kfold_fir_timepoint_separate_sessions'
FIG_DIR = DECODING_DIR / 'fig'
FIG_DIR.mkdir(parents=True, exist_ok=True)
MEASURES = ('fw', 'bw')
COMPONENTS = ('difference', 'contra', 'ipsi', 'midline', 'all_lines')
CONDITIONS = ('early', 'late', 'early_minus_late')
ANALYSES = tuple(
    (measure, component, condition)
    for measure in MEASURES
    for component in COMPONENTS
    for condition in CONDITIONS
)
RESTRICTED_WINDOW = (1.8, 3.8)
FMIN, FMAX = 8.0, 12.0
TW_STRIDE = DECODING_STRIDE = 1
N_PERMUTATIONS = 1000
CLUSTER_ALPHA = CLUSTER_P = 0.05
SEED = 42
restricted_results = {}

for analysis_index, (measure, component, condition) in enumerate(ANALYSES):
    print(
        f'\n=== restricted TW/decoding 1.8–3.8 s: '
        f'{measure.upper()} / {component} / {condition} ===',
        flush=True,
    )
    subjects, tw_time, train_time, test_time, subject_z, matched_counts = (
        cross_corr.build_subject_cross_maps(
            TW_DIR, DECODING_DIR,
            measure=measure, component=component, condition=condition,
            fmin=FMIN, fmax=FMAX,
            tw_limits=RESTRICTED_WINDOW,
            train_limits=RESTRICTED_WINDOW,
            test_limits=RESTRICTED_WINDOW,
            tw_stride=TW_STRIDE, decoding_stride=DECODING_STRIDE,
        )
    )
    mean_z, observed_t, clusters, null_max = cross_corr.cluster_signflip_3d(
        subject_z, permutations=N_PERMUTATIONS,
        cluster_alpha=CLUSTER_ALPHA, cluster_p=CLUSTER_P,
        seed=SEED + analysis_index, return_all=True,
    )
    threshold = student_t.ppf(
        1 - CLUSTER_ALPHA / 2, len(subjects) - 1
    )
    significant = [cluster for cluster in clusters if cluster['significant']]
    cross_corr.print_significant_cluster_ranges(
        observed_t, clusters, tw_time, train_time, test_time
    )
    tag = f'{measure}_{component}_{condition}'
    figure_path = (
        FIG_DIR
        / f'exploratory_tw1p8-3p8_dec1p8-3p8_{tag}_3d-statistic-cluster_shared-origin-v2.png'
    )
    figure = cross_corr.plot_3d_statistic_cluster_panels(
        observed_t, clusters, tw_time, train_time, test_time,
        threshold, figure_path,
        title=(
            f'{measure.upper()} TW ({component}, 1.8–3.8 s) × '
            f'{condition} time-cross decoding [exploratory same-window]'
        ),
    )
    all_cluster_p = sorted(cluster['p'] for cluster in clusters)
    restricted_results[(measure, component, condition)] = {
        'shape': observed_t.shape,
        'n_significant': len(significant),
        'minimum_corrected_p': all_cluster_p[0] if all_cluster_p else None,
        'figure': figure_path,
    }
    print(
        f'significant clusters: {len(significant)}; '
        f'min corrected p: '
        f'{restricted_results[(measure, component, condition)]["minimum_corrected_p"]}',
        flush=True,
    )
    print(f'3-D panel figure saved to: {figure_path}', flush=True)
    del subject_z, mean_z, observed_t, clusters, null_max, figure

print('\nExploratory same-window 3-D summary (30 combinations):')
for key, result in restricted_results.items():
    print(
        key,
        'significant=', result['n_significant'],
        'min_p=', result['minimum_corrected_p'],
        'figure=', result['figure'],
    )

## 3. 三个目标组合的可交互三维 cube

只检验三个预先指定组合：FW difference × early、BW difference × early、BW difference × late。TW 限制为 1.8–3.8 s，decoding training/test 保留 0–6 s。完整 cube 的连续颜色显示全部 voxel 的群体 t 值；黑色线框标出三维 cluster 校正后 p < .05 的范围。此 cell 自带 imports、路径和参数，可在重启 kernel 后单独运行。

In [ ]:
# Standalone cell: safe to run immediately after restarting the kernel.
from pathlib import Path
import importlib
import sys
from scipy.stats import t as student_t

PROJECT_DIR = Path('/home/dilay/project2/tw')
MODULE_DIR = PROJECT_DIR / 'travelling_waves/tw/fft/micheal/corr/cross_corr'
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import michael_tw_cross_correlation as cross_corr
cross_corr = importlib.reload(cross_corr)
assert hasattr(cross_corr, 'plot_3d_statistic_cluster_panels')

TW_DIR = PROJECT_DIR / 'results/micheal_fft'
DECODING_DIR = PROJECT_DIR / 'results/michael_alpha_time_cross_basis_kfold_fir_timepoint_separate_sessions'
FMIN, FMAX = 8.0, 12.0
TW_STRIDE = 1
DECODING_STRIDE = 1
N_PERMUTATIONS = 1000
CLUSTER_ALPHA = CLUSTER_P = 0.05
SEED = 42

TARGET_ANALYSES = (
    ('fw', 'difference', 'early'),
    ('fw', 'difference', 'late'),
    ('bw', 'difference', 'early'),
    ('bw', 'difference', 'late'),
)
INTERACTIVE_DIR = DECODING_DIR / 'fig'
INTERACTIVE_DIR.mkdir(parents=True, exist_ok=True)

def run_target_cube_set(label, tw_limits, train_limits, test_limits):
    summaries = {}
    for analysis_index, (measure, component, condition) in enumerate(TARGET_ANALYSES):
        print(
            f'\n=== {label}: {measure.upper()} / {component} / {condition} ===',
            flush=True,
        )
        subjects, tw_time, train_time, test_time, subject_z, matched_counts = (
            cross_corr.build_subject_cross_maps(
                TW_DIR, DECODING_DIR,
                measure=measure, component=component, condition=condition,
                fmin=FMIN, fmax=FMAX,
                tw_limits=tw_limits, train_limits=train_limits,
                test_limits=test_limits, tw_stride=TW_STRIDE,
                decoding_stride=DECODING_STRIDE,
            )
        )
        mean_z, observed_t, clusters, null_max = cross_corr.cluster_signflip_3d(
            subject_z, permutations=N_PERMUTATIONS,
            cluster_alpha=CLUSTER_ALPHA, cluster_p=CLUSTER_P,
            seed=SEED + analysis_index, return_all=True,
        )
        threshold = student_t.ppf(
            1 - CLUSTER_ALPHA / 2, len(subjects) - 1
        )
        cluster_p_values = sorted(cluster['p'] for cluster in clusters)
        significant = [cluster for cluster in clusters if cluster['significant']]
        cross_corr.print_significant_cluster_ranges(
            observed_t, clusters, tw_time, train_time, test_time
        )
        tag = f'{measure}_{component}_{condition}'
        figure_path = INTERACTIVE_DIR / f'{label}_{tag}_3d-statistic-cluster_shared-origin-v2.png'
        figure = cross_corr.plot_3d_statistic_cluster_panels(
            observed_t, clusters, tw_time, train_time, test_time,
            threshold, figure_path,
            title=(
                f'{measure.upper()} TW ({component}) × {condition} decoding '
                f'[{label}]'
            ),
        )
        summaries[(measure, component, condition)] = {
            'shape': observed_t.shape,
            'n_significant': len(significant),
            'minimum_corrected_p': (
                cluster_p_values[0] if cluster_p_values else None
            ),
            'figure': figure_path,
        }
        print(summaries[(measure, component, condition)], flush=True)
        print(f'3-D panel figure saved to: {figure_path}', flush=True)
        del subject_z, mean_z, observed_t, clusters, null_max, figure
    return summaries

target_cube_results = run_target_cube_set(
    label='tw1p8-3p8_dec0-6',
    tw_limits=(1.8, 4.3),
    train_limits=(1.8, 4.3),
    test_limits=(1.8, 4.3),
)

In [ ]:
from scipy.stats import t as student_t
cross_corr = importlib.reload(cross_corr)
assert hasattr(cross_corr, 'plot_interactive_3d_cube')

TARGET_ANALYSES = (
    ('fw', 'difference', 'early'),
    ('bw', 'difference', 'early'),
    ('bw', 'difference', 'late'),
)
INTERACTIVE_DIR = DECODING_DIR / 'fig'
INTERACTIVE_DIR.mkdir(parents=True, exist_ok=True)

def run_target_cube_set(label, tw_limits, train_limits, test_limits):
    summaries = {}
    for analysis_index, (measure, component, condition) in enumerate(TARGET_ANALYSES):
        print(
            f'\n=== {label}: {measure.upper()} / {component} / {condition} ===',
            flush=True,
        )
        subjects, tw_time, train_time, test_time, subject_z, matched_counts = (
            cross_corr.build_subject_cross_maps(
                TW_DIR, DECODING_DIR,
                measure=measure, component=component, condition=condition,
                fmin=FMIN, fmax=FMAX,
                tw_limits=tw_limits, train_limits=train_limits,
                test_limits=test_limits, tw_stride=TW_STRIDE,
                decoding_stride=DECODING_STRIDE,
            )
        )
        mean_z, observed_t, clusters, null_max = cross_corr.cluster_signflip_3d(
            subject_z, permutations=N_PERMUTATIONS,
            cluster_alpha=CLUSTER_ALPHA, cluster_p=CLUSTER_P,
            seed=SEED + analysis_index, return_all=True,
        )
        threshold = student_t.ppf(
            1 - CLUSTER_ALPHA / 2, len(subjects) - 1
        )
        cluster_p_values = sorted(cluster['p'] for cluster in clusters)
        significant = [cluster for cluster in clusters if cluster['significant']]
        tag = f'{measure}_{component}_{condition}'
        html_path = INTERACTIVE_DIR / f'{label}_{tag}_interactive-cube.html'
        figure = cross_corr.plot_interactive_3d_cube(
            observed_t, clusters, tw_time, train_time, test_time,
            threshold, html_path,
            title=(
                f'{measure.upper()} TW ({component}) × {condition} decoding '
                f'[{label}]'
            ),
        )
        summaries[(measure, component, condition)] = {
            'shape': observed_t.shape,
            'n_significant': len(significant),
            'minimum_corrected_p': (
                cluster_p_values[0] if cluster_p_values else None
            ),
            'html': html_path,
        }
        print(summaries[(measure, component, condition)], flush=True)
        display(figure)
        del subject_z, mean_z, observed_t, clusters, null_max, figure
    return summaries

target_cube_results = run_target_cube_set(
    label='tw1p8-3p8_dec0-6',
    tw_limits=(1.8, 3.8),
    train_limits=(0.0, 6.0),
    test_limits=(0.0, 6.0),
)

## 探索性 off-diagonal-only 3D correlation（全部 TW components，可单独运行）

在 1.8–3.8 s 的同窗空间内，探索性检验 FW/BW × `difference`、`contra`、`ipsi`、`midline`、`all_lines` × early、late、early−late 的全部组合。在三维 cluster sign-flip 前排除 `Training time = Test time` 主对角线；对角线固定为 0，不能形成显著 voxel，也不能连接对角线两侧的 cluster。由于同时考察多个方向、component 和 decoding condition，这些结果应明确作为探索性分析报告。

In [ ]:
# Standalone off-diagonal-only 3-D correlation cell.
from pathlib import Path
import importlib
import sys
import numpy as np
from scipy.stats import t as student_t

PROJECT_DIR = Path('/home/dilay/project2/tw')
MODULE_DIR = PROJECT_DIR / 'travelling_waves/tw/fft/micheal/corr/cross_corr'
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import michael_tw_cross_correlation as cross_corr
cross_corr = importlib.reload(cross_corr)

TW_DIR = PROJECT_DIR / 'results/micheal_fft'
DECODING_DIR = PROJECT_DIR / 'results/michael_alpha_time_cross_basis_kfold_fir_timepoint_separate_sessions'
FIGURE_DIR = DECODING_DIR / 'fig'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MEASURES = ('fw', 'bw')
COMPONENTS = ('difference', 'contra', 'ipsi', 'midline', 'all_lines')
CONDITIONS = ('early', 'late', 'early_minus_late')
TARGET_ANALYSES = tuple(
    (measure, component, condition)
    for measure in MEASURES
    for component in COMPONENTS
    for condition in CONDITIONS
)
TW_LIMITS = TRAIN_LIMITS = TEST_LIMITS = (1.8, 4.3)
FMIN, FMAX = 8.0, 12.0
N_PERMUTATIONS = 1000
CLUSTER_ALPHA = CLUSTER_P = 0.05
SEED = 42
OFF_DIAGONAL_MIN_GAP_S = 0.0  # 0 excludes only train == test
off_diagonal_results = {}

for analysis_index, (measure, component, condition) in enumerate(TARGET_ANALYSES):
    print(f'\n=== OFF-DIAGONAL: {measure.upper()} / {component} / {condition} ===', flush=True)
    subjects, tw_time, train_time, test_time, subject_z, matched_counts = cross_corr.build_subject_cross_maps(
        TW_DIR, DECODING_DIR, measure=measure, component=component,
        condition=condition, fmin=FMIN, fmax=FMAX,
        tw_limits=TW_LIMITS, train_limits=TRAIN_LIMITS, test_limits=TEST_LIMITS,
        tw_stride=1, decoding_stride=1,
    )
    separation = np.abs(train_time[:, None] - test_time[None, :])
    tolerance = 0.25 * min(np.median(np.diff(train_time)), np.median(np.diff(test_time)))
    off_diagonal_2d = separation > (OFF_DIAGONAL_MIN_GAP_S + tolerance)
    subject_z = np.where(off_diagonal_2d[None, None, :, :], subject_z, 0.0).astype(np.float32, copy=False)
    print(f'  retained {off_diagonal_2d.sum()}/{off_diagonal_2d.size} train×test cells', flush=True)
    mean_z, observed_t, clusters, null_max = cross_corr.cluster_signflip_3d(
        subject_z, permutations=N_PERMUTATIONS, cluster_alpha=CLUSTER_ALPHA,
        cluster_p=CLUSTER_P, seed=SEED + analysis_index, return_all=True,
    )
    threshold = student_t.ppf(1 - CLUSTER_ALPHA / 2, len(subjects) - 1)
    significant = [cluster for cluster in clusters if cluster['significant']]
    cluster_p_values = sorted(cluster['p'] for cluster in clusters)
    cross_corr.print_significant_cluster_ranges(observed_t, clusters, tw_time, train_time, test_time)
    tag = f'{measure}_{component}_{condition}'
    figure_path = FIGURE_DIR / f'exploratory_tw1p8-3p8_dec1p8-3p8_off-diagonal_{tag}_3d-statistic-cluster_shared-origin-v2.png'
    figure = cross_corr.plot_3d_statistic_cluster_panels(
        observed_t, clusters, tw_time, train_time, test_time, threshold, figure_path,
        title=f'{measure.upper()} TW ({component}) × {condition} decoding [exploratory off-diagonal only, 1.8–3.8 s]',
    )
    result = {
        'shape': observed_t.shape,
        'off_diagonal_cells': int(off_diagonal_2d.sum()),
        'n_significant': len(significant),
        'minimum_corrected_p': cluster_p_values[0] if cluster_p_values else None,
        'figure': figure_path,
    }
    off_diagonal_results[(measure, component, condition)] = result
    print(result, flush=True)
    print(f'3-D off-diagonal figure saved to: {figure_path}', flush=True)
    del subject_z, mean_z, observed_t, clusters, null_max, figure

print('\nExploratory off-diagonal-only summary (30 combinations):')
for key, result in off_diagonal_results.items():
    print(key, result)


## Diagonal decoding × TW correlation（二维，可单独运行）

仅提取每个 trial 的 `Training time = Test time` decoding evidence，计算 `TW time × diagonal decoding time` 的二维 trialwise correlation，并进行二维 cluster-mass sign-flip 校正。

In [ ]:
# Standalone 2-D diagonal decoding × TW correlation.
from pathlib import Path
import importlib
import sys
from scipy.stats import t as student_t

PROJECT_DIR = Path('/home/dilay/project2/tw')
MODULE_DIR = PROJECT_DIR / 'travelling_waves/tw/fft/micheal/corr/cross_corr'
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import michael_tw_cross_correlation as cross_corr
cross_corr = importlib.reload(cross_corr)

TW_DIR = PROJECT_DIR / 'results/micheal_fft'
DECODING_DIR = PROJECT_DIR / 'results/michael_alpha_time_cross_basis_kfold_fir_timepoint_separate_sessions'
FIGURE_DIR = DECODING_DIR / 'fig'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TARGET_ANALYSES = (
    ('fw', 'difference', 'early'),
    ('fw', 'difference', 'late'),
    ('bw', 'difference', 'early'),
    ('bw', 'difference', 'late'),
)
WINDOW = (1.8, 4.3)
FMIN, FMAX = 8.0, 12.0
N_PERMUTATIONS = 1000
CLUSTER_ALPHA = CLUSTER_P = 0.05
SEED = 242
diagonal_tw_results = {}

for analysis_index, (measure, component, condition) in enumerate(TARGET_ANALYSES):
    print(f'\n=== DIAGONAL 2-D: {measure.upper()} / {component} / {condition} ===', flush=True)
    subjects, tw_time, train_time, test_time, subject_z, matched_counts = cross_corr.build_subject_cross_maps(
        TW_DIR, DECODING_DIR, measure=measure, component=component,
        condition=condition, fmin=FMIN, fmax=FMAX,
        tw_limits=WINDOW, train_limits=WINDOW, test_limits=WINDOW,
        tw_stride=1, decoding_stride=1, diagonal_only=True,
    )
    decoding_time = train_time
    print(f'  2-D subject map: TW {tw_time.size} × diagonal decoding {decoding_time.size}', flush=True)
    mean_z, observed_t, clusters, null_max = cross_corr.cluster_signflip_2d(
        subject_z, permutations=N_PERMUTATIONS, cluster_alpha=CLUSTER_ALPHA,
        cluster_p=CLUSTER_P, seed=SEED + analysis_index, return_all=True,
    )
    significant = [cluster for cluster in clusters if cluster['significant']]
    cluster_p_values = sorted(cluster['p'] for cluster in clusters)
    cross_corr.print_significant_cluster_ranges_2d(
        observed_t, clusters, tw_time, decoding_time
    )
    tag = f'{measure}_{component}_{condition}'
    figure_path = FIGURE_DIR / f'tw1p8-4p3_diagonal-decoding_{tag}_2d-cluster.png'
    figure = cross_corr.plot_diagonal_tw_correlation_2d(
        observed_t, clusters, tw_time, decoding_time, figure_path,
        title=f'{measure.upper()} TW ({component}) × {condition} diagonal decoding',
    )
    result = {
        'shape': observed_t.shape,
        'n_significant': len(significant),
        'minimum_corrected_p': cluster_p_values[0] if cluster_p_values else None,
        'figure': figure_path,
    }
    diagonal_tw_results[(measure, component, condition)] = result
    print(result, flush=True)
    print(f'2-D diagonal figure saved to: {figure_path}', flush=True)
    del subject_z, mean_z, observed_t, clusters, null_max, figure

print('\nDiagonal decoding × TW summary:')
for key, result in diagonal_tw_results.items():
    print(key, result)


## FW/BW contra−ipsi 时间线（1.8–4.3 s，可单独运行）

按原始 `Results_header` 的编码计算：`cue_loc=1` 表示左侧项目先测（early-left），所以右半球为 contra；`cue_loc=2` 表示右侧项目先测，所以左半球为 contra。TW 时间使用 250-sample 滑窗的中心：`-1.25 + (start + 125) / 500` 秒。细线为每位被试，粗线及阴影为组均值 ± SEM。

In [ ]:
# Standalone: subject and group FW/BW contra-minus-ipsi time courses.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path('/home/dilay/project2/tw')
TW_DIR = PROJECT_DIR / 'results/micheal_fft'
FIGURE_DIR = PROJECT_DIR / 'results/michael_alpha_time_cross_basis_kfold_fir_timepoint_separate_sessions/fig'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
WINDOW_S = (1.8, 4.3)
FMIN, FMAX = 8.0, 12.0
SUBJECTS = range(1, 20)

subject_curves = {'fw': [], 'bw': []}
plot_time_s = None
for subject in SUBJECTS:
    session_curves = {'fw': [], 'bw': []}
    session_weights = []
    for session in (1, 2):
        path = TW_DIR / f'subj{subject:02d}_sess{session}.pkl'
        with path.open('rb') as stream:
            data = pickle.load(stream)
        # Saved time/starts are window-start sample indices, not milliseconds.
        # EEG begins at -1.25 s; each TW window is 250 samples at 500 Hz.
        starts = np.asarray(data.get('starts', data['time']), dtype=float)
        time_s = -1.25 + (starts + 125) / 500.0
        tolerance = 1e-9  # retain nominal 1.8/4.3-s samples despite float rounding
        time_mask = (time_s >= WINDOW_S[0] - tolerance) & (time_s <= WINDOW_S[1] + tolerance)
        freq = np.asarray(data['ff'], dtype=float)
        freq_mask = (freq >= FMIN) & (freq <= FMAX)
        cue_left = np.asarray(data['cue_loc']).reshape(-1) == 2
        good = ~np.asarray(data.get('is_bad_epoch', np.zeros(cue_left.size, dtype=bool)), dtype=bool)
        if plot_time_s is None:
            plot_time_s = time_s[time_mask]
        elif not np.allclose(plot_time_s, time_s[time_mask]):
            raise ValueError(f'TW time mismatch in {path.name}')
        for measure in ('fw', 'bw'):
            actual = np.asarray(data[f'{measure}max'], dtype=float)
            surrogate = np.asarray(data[f'{measure}ssmax'], dtype=float)
            db = 10.0 * np.log10(actual / surrogate)
            physical_left = db[0:5].mean(axis=0)
            physical_right = db[6:11].mean(axis=0)
            contra = np.where(cue_left[None, None, :], physical_right, physical_left)
            ipsi = np.where(cue_left[None, None, :], physical_left, physical_right)
            difference = contra - ipsi
            curve = difference[freq_mask][:, time_mask, :][:, :, good].mean(axis=(0, 2))
            session_curves[measure].append(curve)
        session_weights.append(int(good.sum()))
        del data, actual, surrogate, db, physical_left, physical_right, contra, ipsi, difference
    for measure in ('fw', 'bw'):
        subject_curves[measure].append(np.average(session_curves[measure], axis=0, weights=session_weights))
    print(f'subject {subject:02d}: n_good={sum(session_weights)}', flush=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True, sharey=True, constrained_layout=True)
styles = {'fw': ('#2878B5', 'Forward TW'), 'bw': ('#D1495B', 'Backward TW')}
for ax, measure in zip(axes, ('fw', 'bw')):
    curves = np.asarray(subject_curves[measure])
    mean = curves.mean(axis=0)
    sem = curves.std(axis=0, ddof=1) / np.sqrt(curves.shape[0])
    color, label = styles[measure]
    for subject_curve in curves:
        ax.plot(plot_time_s, subject_curve, color=color, lw=0.75, alpha=0.22)
    ax.fill_between(plot_time_s, mean-sem, mean+sem, color=color, alpha=0.22, linewidth=0)
    ax.plot(plot_time_s, mean, color=color, lw=2.8, label='Group mean ± SEM')
    ax.axhline(0, color='0.25', ls='--', lw=1)
    ax.set(title=f'{label}: contra − ipsi', xlabel='TW time (s)', xlim=WINDOW_S)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, loc='best')
axes[0].set_ylabel('Alpha TW strength difference (dB)')
fig.suptitle('Micheal: alpha (8–12 Hz) TW lateralization; cue_loc 1 = early-left', fontsize=14)
figure_path = FIGURE_DIR / 'michael_fw_bw_difference_timecourses_1p8-4p3.png'
fig.savefig(figure_path, dpi=250, bbox_inches='tight')
print(f'Saved: {figure_path}', flush=True)
plt.show()
